# Part 1 Handwritten Digit Recognition Using LeNet-5: Overview And Software Implementation

## 1. Overview

LeNet-5 is one of the most fundamental convolution neural networks(CNNs). It was originally designed to recognize handwritten digits and achieved remarkable success on the MNIST dataset. The network features a hierarchical design with convolutional layers to extract features, followed by pooling layers to reduce dimensionality, and finally fully connected layers for classification. 

In this project-based lab, we would implement a accelerator to accelerate a simplified LeNet-5, to classify MNIST images(recognize the handwritten digit). The goal is to enhance the computation speed for a simplified LeNet-5 with int2 quantization.


The architecture of our simplified LeNet-5 are as follows:

<p align="center"> Simplified LeNet-5 Model</p>

| Layer                                                 | Type    | Output Dimensions | Kernel Size | Stride | Padding | Number of Parameters |
|:-----------------------------------------------------:|:-------:|:-----------------:|:-----------:|:------:|:-------:|:--------------------:|
| input         | -       | 28x28x1           | -           | -      | -       | -                    |
| conv0 | Conv2D  | 24x24x16          | 5x5         | 1      | None    | (5x5x1+1)x16=416     |
| maxpool0        | MaxPool | 12x12x16          | 2x2         | 2      | -       | 0                    |
| conv1 | Conv2D  | 8x8x16            | 5x5         | 1      | None    | (5x5x16+1)x16=6416   |
| maxpool1        | MaxPool | 4x4x16            | 2x2         | 2      | -       | 0                    |
| flatten          | Flatten | 256               | -           | -      | -       | 0                    |
| matmul0          | Dense   | 10                | -           | -      | -       | 256x10+10=2570       |

<p align="center">Total: 9402 parameters</p>

Data flow graph:

![data flow graph](./image/lenet5_dfg.png)


In the previous conv_filter chapter, we have introduced the concept and Vitis HLS implementation of Convolution Filter. In this Chapter, we will continue to apply data-level and task-level optimizations to accelerate LeNet-5.

## 2. Software Implementation of LeNet-5

In this section, we will explore the software implementation of the simplified LeNet-5 model. The implementation will focus on the key components outlined in the architecture table above:

1. Two convolution layers (conv0, conv1) for feature extraction
2. Two max pooling layers for dimensionality reduction 
3. A flatten layer to transform the 3D feature maps into a 1D vector
4. A fully connected layer for final classification

This implementation serves as the baseline for our subsequent hardware acceleration efforts. By understanding the software implementation first, we can better identify opportunities for optimization when moving to hardware. We will use Python Numpy library to implement the software of LeNet-5 and run it on the CPU of AMD-Xilinx FPGA, and compare its performance with Vitis HLS implementation.  


First of all, we will read inputs and weights from files and convert them to numpy arrays.

In [1]:
# read inputs and weights from files and convert them to numpy arrays

from PIL import Image
import numpy as np

def image_to_binary_array(image_path, threshold=128):
    # Open and convert image to grayscale
    img = Image.open(image_path).convert('L')
    
    # Ensure image is 28x28
    if img.size != (28, 28):
        img = img.resize((28, 28))
    
    # Convert to numpy array
    pixel_array = np.array(img)
    
    # Create binary array (1 for white-ish pixels, 0 for dark pixels)
    binary_array = (pixel_array > threshold).astype(int)
    
    # Convert each row to a single integer (28 bits)
    input_array = []
    for row in binary_array:
        binary_num = 0
        for bit in row:
            binary_num = (binary_num << 1) | bit
        input_array.append(binary_num)
    
    return input_array, binary_array

# Example usage
image_path = "./image/mnist_image.png"
# input_array is a list of 28 integers, each representing a row of the image
# binary_array is a 28x28 numpy array, each element is 1 or 0
input_array, binary_array = image_to_binary_array(image_path)

for i, num in enumerate(input_array):
    print(f"0b{bin(num)[2:].zfill(28)}")

# Add an extra dimension for the channel (required for convolution)
binary_array = binary_array.reshape(28, 28, 1)


# Define the path to the file
file_path1 = 'conv0_weight.txt'

# Read the file content
with open(file_path1, 'r') as file:
    lines = file.readlines()

# Extract the array content between the braces
array_content = ' '.join(lines).split('{')[1].split('}')[0]

# Clean the content, filter out empty strings, and convert to integers
array_values = list(map(int, filter(None, array_content.replace('\n', '').replace(' ', '').split(','))))

# Convert to a NumPy array and reshape it to the desired shape (16, 5, 5, 1)
conv0_weight = np.array(array_values).reshape(16, 5, 5, 1)

# Print the NumPy array to verify the result
# print(array_values)

# Define the path to the file
file_path2 = 'conv1_weight.txt'

# Read the file content
with open(file_path2, 'r') as file:
    lines = file.readlines()

# Extract the array content between the braces
array_content = ''.join(lines).split('{')[1].split('}')[0]

# Clean the content, filter out empty strings, and convert to integers
array_values = list(map(int, filter(None, array_content.replace('\n', '').replace(' ', '').split(','))))

# Convert to a NumPy array and reshape it to the desired shape (16, 5, 5, 16)
conv1_weight = np.array(array_values).reshape(16, 5, 5, 16)

# Print the NumPy array to verify the result
# print(array_values)

# Define the path to the file
file_path3 = 'matmul0_weight.txt'

# Read the file content
with open(file_path3, 'r') as file:
    lines = file.readlines()

# Extract the array content between the braces
array_content = ''.join(lines).split('{')[1].split('}')[0]

# Clean the content, filter out empty strings, and convert to integers
array_values = list(map(int, filter(None, array_content.replace('\n', '').replace(' ', '').split(','))))

# Convert to a NumPy array and reshape it to the desired shape (10, 256)
matmul0_weight = np.array(array_values).reshape(10, 256)

# Print the NumPy array to verify the result
# print(matmul0_weight)

0b0000000000000000000000000000
0b0000000000000000000000000000
0b0000000000000000000000000000
0b0000000000000000000000000000
0b0000000000000000000000000000
0b0000000000000000001000000000
0b0000000000000000011000000000
0b0000000000000000011000000000
0b0000000000000000111000000000
0b0000000000100000110000000000
0b0000000000100000110000000000
0b0000000000100000110000000000
0b0000000000100001110000000000
0b0000000001100001100000000000
0b0000000001111111100000000000
0b0000000001111111000000000000
0b0000000011100011000000000000
0b0000000000000011000000000000
0b0000000000000110000000000000
0b0000000000000110000000000000
0b0000000000000110000000000000
0b0000000000001110000000000000
0b0000000000001110000000000000
0b0000000000001100000000000000
0b0000000000001100000000000000
0b0000000000000000000000000000
0b0000000000000000000000000000
0b0000000000000000000000000000


Then we'll implement LeNet-5 using the Python NumPy library. Since Python doesn't have an int2 data type and the processor on AMD-Xilinx FPGA doesn't support int2, we'll use the standard int type for this implementation.


In [3]:
def conv2d(input_data, weight, stride=1):
    input_height, input_width = input_data.shape[0], input_data.shape[1]
    kernel_size = weight.shape[1]
    num_filters = weight.shape[0]
    
    # Calculate output dimensions
    output_height = (input_height - kernel_size) // stride + 1
    output_width = (input_width - kernel_size) // stride + 1
    
    # Initialize output
    output = np.zeros((output_height, output_width, num_filters))
    
    # Perform convolution
    for f in range(num_filters):
        for i in range(0, output_height, stride):
            for j in range(0, output_width, stride):
                output[i, j, f] = np.sum(
                    input_data[i:i+kernel_size, j:j+kernel_size] * weight[f, :, :, :]
                )
    return output

def maxpool2d(input_data, pool_size=2, stride=2):
    input_height, input_width, channels = input_data.shape
    
    # Calculate output dimensions
    output_height = input_height // stride
    output_width = input_width // stride
    
    # Initialize output
    output = np.zeros((output_height, output_width, channels))
    
    # Perform max pooling
    for c in range(channels):
        for i in range(output_height):
            for j in range(output_width):
                output[i, j, c] = np.max(
                    input_data[i*stride:i*stride+pool_size, 
                             j*stride:j*stride+pool_size, c]
                )
    return output

def lenet5_forward(input_image):
    # Reshape input to match expected dimensions (28x28x1)
    x = np.array(input_image).reshape(28, 28, 1)
    
    # First convolution layer
    conv0_output = conv2d(x, conv0_weight)  # Output: 24x24x16
    
    # First max pooling layer
    pool0_output = maxpool2d(conv0_output)  # Output: 12x12x16
    
    # Second convolution layer
    conv1_output = conv2d(pool0_output, conv1_weight)  # Output: 8x8x16
    
    # Second max pooling layer
    pool1_output = maxpool2d(conv1_output)  # Output: 4x4x16
    
    # Flatten
    flatten_output = pool1_output.reshape(-1)  # Output: 256
    
    # Fully connected layer (matrix multiplication)
    output = np.dot(matmul0_weight, flatten_output)  # Output: 10
    
    return output

# Test the model with the loaded image
test_input = np.array(binary_array)
import time
start = time.time()
result = lenet5_forward(test_input)
end = time.time()

print("Time taken for inference: {}s".format((end-start) / 10000))

# Print the predicted digit (index of maximum value)
predicted_digit = np.argmax(result)
print(f"Predicted digit: {predicted_digit}")
print(f"Confidence scores: {result}")

Time taken for inference: 0.00019597129821777344s
Predicted digit: 4
Confidence scores: [-1453.   302.  -648.  -358.   967.  -845.  -423.  -832.   134.    29.]


Hence, we can conclude that the time taken to perform simplified LeNet-5 Inference in Numpy is approximately 0.00020 seconds. 

---------------------------------------
<p align="center">Copyright&copy; 2025 Advanced Micro Devices</p>